# Fine-tune no Qwen-Coder **1.5B** — vários experts por sessão

O `finetune_expert_colab.ipynb` treina um expert por vez, o que fazia sentido no 7B: cada
rodada levava ~13 min e consumia quase toda a T4. No 1.5B a rodada cai para ~4-5 min, e
repetir oito células dezoito vezes vira o gargalo. Aqui é uma **fila**.

**Por que o 1.5B.** Medido na máquina onde isso vai rodar (CPU, sem GPU utilizável):

| | entrada | prefill | total |
|---|---:|---:|---:|
| axon-go (7B) | 283 tok | 55,4 s | 105 s |
| qwen 1.5B | 283 tok | **10,8 s** | **48 s** |

Cinco vezes mais rápido para ler o contexto. O que se perde é profundidade e concisão --
o 1.5B respondeu em lista genérica onde o 7B foi direto ao ponto. É exatamente isso que
o adapter deve recuperar, e é a pergunta que esta sessão responde.

**As pastas são separadas por base**, então nada aqui sobrescreve o que você treinou no 7B:

```
MyDrive/axon_lora/qwen/<expert>/        ← o 7B, intacto
MyDrive/axon_lora/qwen-1.5b/<expert>/   ← o que sai daqui
MyDrive/axon_gguf/qwen-1.5b/            ← os GGUF convertidos
```

> `Ambiente de execução → Alterar o tipo de ambiente → T4 GPU`. O treino exige CUDA:
> o Unsloth e a quantização de 4 bits não rodam em CPU.

## 1. Instalar

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

# O Unsloth exige protobuf 5.x; o Colab vem com uma anterior. Atualizar ANTES de
# qualquer import que carregue protobuf -- depois de carregado, só reiniciando a sessão.
!pip install -q -U "protobuf>=5.28"

import torch
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Clonar o repo

In [ ]:
%cd /content
import os, sys, gc, json, time

if os.path.isdir("axon-llm"):
    !git -C axon-llm pull --quiet && echo "repo atualizado"
else:
    !git clone --depth 1 https://github.com/geraldogrise/axon-llm.git axon-llm

sys.path.insert(0, "/content/axon-llm/notebooks")
import importlib
import axon_colab as ac
importlib.reload(ac)

from google.colab import drive
drive.mount("/content/drive")

## 3. A fila

Comece por dois. A pergunta desta sessão é **se o adapter no 1.5B recupera a qualidade
que o 7B tinha** — e dois experts respondem isso em 10 minutos. Se recuperar, você volta
e roda os 18; se não, economizou uma hora e meia.

`ac.ORDEM_FINAL` tem os 18 na ordem certa, se quiser tudo de uma vez.

In [ ]:
BASE = "qwen-1.5b"
FILA = ["go", "docker"]          # ac.ORDEM_FINAL para todos os 18

MAX_LEN = 2048
EPOCAS = 2

print(f"base : {ac.BASES[BASE]}")
print(f"fila : {FILA}")
for n in FILA:
    ac.check(n)                   # falha agora se algum nome estiver errado
print("nomes válidos")

## 4. Treinar a fila

Cada expert recarrega o modelo base do zero. Isso é obrigatório, não desperdício: sem
recarregar, o segundo expert treinaria **por cima** do adapter do primeiro e os dois
sairiam misturados. No 1.5B a recarga custa segundos.

Um expert que falhar não derruba a fila -- o erro é registrado e ela segue.

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

resultados, erros = [], []

for i, expert in enumerate(FILA, 1):
    print(f"\n{'=' * 60}\n[{i}/{len(FILA)}] {expert}\n{'=' * 60}", flush=True)
    t0 = time.time()
    try:
        _, _, _, _, rotulo = ac.check(expert)
        LORA = ac.lora_dir(BASE, expert)
        CKPT = ac.drive_dir("axon_ckpt", BASE, expert)

        # modelo limpo a cada expert
        modelo, tokenizer = FastLanguageModel.from_pretrained(
            model_name=ac.BASES[BASE], max_seq_length=MAX_LEN, load_in_4bit=True)
        modelo = FastLanguageModel.get_peft_model(
            modelo, r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            use_gradient_checkpointing="unsloth", random_state=42)

        dados = ac.fetch_data(expert)
        exemplos = ac.sft_examples(dados, tokenizer, rotulo, max_chars=6000)
        ds = Dataset.from_list(exemplos).train_test_split(test_size=0.05, seed=42)
        print(f"{len(exemplos)} exemplos", flush=True)

        trainer = SFTTrainer(
            model=modelo, tokenizer=tokenizer,
            train_dataset=ds["train"], eval_dataset=ds["test"],
            args=SFTConfig(
                dataset_text_field="text", max_seq_length=MAX_LEN,
                # o 1.5B cabe folgado na T4: lote maior, menos passos
                per_device_train_batch_size=4, gradient_accumulation_steps=2,
                num_train_epochs=EPOCAS, warmup_steps=10, learning_rate=2e-4,
                optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine",
                logging_steps=10, seed=42, report_to="none",
                output_dir=CKPT, save_steps=50, save_total_limit=1))

        retomar = ac.ultimo_checkpoint(CKPT)
        trainer.train(resume_from_checkpoint=retomar)

        modelo.save_pretrained(LORA)
        tokenizer.save_pretrained(LORA)

        # O NotebookProgressCallback quebra o evaluate avulso; tirar não muda a métrica.
        try:
            from transformers.utils.notebook import NotebookProgressCallback
            trainer.remove_callback(NotebookProgressCallback)
        except Exception:
            pass

        m = trainer.evaluate()
        m.update({"base": BASE, "modelo": ac.BASES[BASE], "expert": expert,
                  "exemplos": len(exemplos), "minutos": round((time.time() - t0) / 60, 1)})
        with open(os.path.join(LORA, "metrica.json"), "w", encoding="utf-8") as f:
            json.dump(m, f, ensure_ascii=False, indent=2)

        resultados.append(m)
        print(f"OK {expert}: eval_loss {m.get('eval_loss', 0):.4f} "
              f"em {m['minutos']} min", flush=True)
    except Exception as erro:
        erros.append((expert, f"{type(erro).__name__}: {erro}"))
        print(f"FALHOU {expert}: {erro}", flush=True)
    finally:
        for nome in ("trainer", "modelo", "tokenizer"):
            if nome in dir():
                exec(f"del {nome}")
        gc.collect()
        torch.cuda.empty_cache()

print(f"\n{'=' * 60}")
for m in resultados:
    print(f"  {m['expert']:<12} eval_loss {m.get('eval_loss', 0):>7.4f}  "
          f"{m['minutos']:>4} min")
for e, msg in erros:
    print(f"  {e:<12} FALHOU: {msg[:60]}")

## 5. Comparar com o 7B

Os dois treinos gravaram `metrica.json` em pastas separadas, então dá para ler os dois.

**Cuidado ao interpretar:** `eval_loss` entre modelos de tamanhos diferentes não é
comparável de forma limpa -- vocabulário e capacidade mudam a escala. Serve para ver se o
treino do 1.5B foi saudável (loss caindo), não para declarar que um modelo é melhor.
Quem responde isso é a leitura das respostas, na célula 7.

In [ ]:
import glob

print(f"{'base':<12} {'expert':<12} {'eval_loss':>10} {'exemplos':>9} {'min':>6}")
for fp in sorted(glob.glob("/content/drive/MyDrive/axon_lora/*/*/metrica.json")):
    m = json.load(open(fp, encoding="utf-8"))
    print(f"{m['base']:<12} {m['expert']:<12} {m.get('eval_loss', 0):>10.4f} "
          f"{m.get('exemplos', 0):>9} {m.get('minutos', '—'):>6}")

## 6. Exportar para o Ollama

Converte cada adapter da fila para GGUF, em `axon_gguf/qwen-1.5b/`. Na sua máquina o
`FROM` do Modelfile passa a ser `qwen2.5-coder:1.5b` — **não** o 7B: adapter só encaixa
na arquitetura em que foi treinado.

In [ ]:
BASE_HF = {"qwen-1.5b": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
           "qwen-3b":   "Qwen/Qwen2.5-Coder-3B-Instruct"}[BASE]

!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_lora_to_gguf.txt

from huggingface_hub import snapshot_download

cfg = snapshot_download(BASE_HF, allow_patterns=["config.json", "tokenizer*", "*.json"])
destino = ac.drive_dir("axon_gguf", BASE)

for expert in FILA:
    lora = ac.lora_dir(BASE, expert)
    saida = f"{destino}/{expert}-lora.gguf"
    !python /content/llama.cpp/convert_lora_to_gguf.py --base {cfg} --outtype f16 --outfile {saida} {lora}
    if os.path.exists(saida):
        print(f"  {expert}: {os.path.getsize(saida) / 1e6:.0f} MB")
    else:
        print(f"  {expert}: FALHOU")

print(f"\nNa sua máquina:")
print(f"  ollama pull qwen2.5-coder:1.5b")
print(f"  Modelfile:  FROM qwen2.5-coder:1.5b")
print(f"              ADAPTER ./<expert>-lora.gguf")
print(f"  ollama create axon15-<expert> -f Modelfile")

## 7. A pergunta que decide

Compare a mesma resposta nos três: 1.5B puro, 1.5B com adapter, e o 7B com adapter.
É a leitura, não o `eval_loss`, que diz se vale treinar os outros dezesseis.

O 1.5B puro, na medição de referência, respondeu *"1. Criar Canais. 2. Enviar e Receber
Dados. 3. Usar select"* — correto e raso. O 7B com adapter respondeu *"o select espera em
vários canais simultaneamente e executa o caso cujo canal estiver pronto; com default ele
não bloqueia"* — direto e específico. Se o adapter no 1.5B puxar a resposta para o
segundo tipo, valeu.

In [ ]:
FastLanguageModel.for_inference(modelo)

def responder(pergunta, n=200):
    ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": pergunta}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    saida = modelo.generate(input_ids=ids, max_new_tokens=n, temperature=0.3,
                            do_sample=True)
    return tokenizer.decode(saida[0][ids.shape[1]:], skip_special_tokens=True)

print(responder("Em Go, como uso select com canais? Responda de forma direta."))

## Se valer a pena

Volte na célula 3 e troque a fila:

```python
FILA = ac.ORDEM_FINAL      # os 18
```

A ~5 min cada, é cerca de uma hora e meia. A fila pula nada e regrava tudo, então rode
só os que faltam se já tiver feito alguns:

```python
FILA = [e for e in ac.ORDEM_FINAL if e not in ("go", "docker")]
```